In [ ]:
import numpy as np
from PIL import Image
import trimesh
import matplotlib.pyplot as plt

input_image_path = ''
output_stl_path = ''
desired_width_cm = 10.0
desired_height_cm = 10.0
extrude_height_cm = 0.2

def load_and_binarize(image_path):
    image = Image.open(image_path).convert('L')
    image_np = np.array(image)
    binary = np.where(image_np < 128, 1, 0)
    return binary

binary_image = load_and_binarize(input_image_path)

plt.figure(figsize=(6,6))
plt.imshow(binary_image, cmap='gray', extent=(0, desired_width_cm, 0, desired_height_cm))
plt.axis('off')
plt.show()

img_height, img_width = binary_image.shape
scale_x = desired_width_cm / img_width
scale_y = desired_height_cm / img_height

black_pixels = np.argwhere(binary_image == 1)

points_2d = [(x * scale_x, (img_height - y) * scale_y) for y, x in black_pixels]


cube_size = min(scale_x, scale_y) 

meshes = []

for (x, y) in points_2d:
    vertices = [
        [x - cube_size/2, y - cube_size/2, 0],
        [x + cube_size/2, y - cube_size/2, 0],
        [x + cube_size/2, y + cube_size/2, 0],
        [x - cube_size/2, y + cube_size/2, 0],
        [x - cube_size/2, y - cube_size/2, extrude_height_cm],
        [x + cube_size/2, y - cube_size/2, extrude_height_cm],
        [x + cube_size/2, y + cube_size/2, extrude_height_cm],
        [x - cube_size/2, y + cube_size/2, extrude_height_cm],
    ]
    faces = [
        [0,1,2], [0,2,3],
        [4,5,6], [4,6,7],
        [0,1,5], [0,5,4],
        [1,2,6], [1,6,5],
        [2,3,7], [2,7,6],
        [3,0,4], [3,4,7],
    ]
    cube = trimesh.Trimesh(vertices=vertices, faces=faces)
    meshes.append(cube)
    if len(meshes) > 1000:
        combined = trimesh.util.concatenate(meshes)
        meshes = [combined]

if len(meshes) > 1:
    combined_mesh = trimesh.util.concatenate(meshes)
else:
    combined_mesh = meshes[0]

combined_mesh.merge_vertices()

combined_mesh.show()

combined_mesh.export(output_stl_path)


In [ ]:
import numpy as np
from PIL import Image
import trimesh
import os

current_directory = r''

input_images = [f for f in os.listdir(current_directory) if f.endswith('.png')]

desired_width_cm  = 13.5
desired_height_cm = 13.5
extrude_height_cm = 0.2

def load_and_binarize(image_path):
    image = Image.open(image_path).convert('L')
    image_np = np.array(image)
    return np.where(image_np < 128, 1, 0)

for filename in input_images:
    img_path = os.path.join(current_directory, filename)
    binary_image = load_and_binarize(img_path)

    img_height, img_width = binary_image.shape
    scale_x = desired_width_cm  / img_width
    scale_y = desired_height_cm / img_height

    black_pixels = np.argwhere(binary_image == 1)
    points_2d = [(x * scale_x, (img_height - y) * scale_y) for y, x in black_pixels]

    cube_size = min(scale_x, scale_y)
    meshes = []

    for (x, y) in points_2d:
        z0, z1 = 0, extrude_height_cm
        vertices = [
            [x - cube_size/2, y - cube_size/2, z0],
            [x + cube_size/2, y - cube_size/2, z0],
            [x + cube_size/2, y + cube_size/2, z0],
            [x - cube_size/2, y + cube_size/2, z0],
            [x - cube_size/2, y - cube_size/2, z1],
            [x + cube_size/2, y - cube_size/2, z1],
            [x + cube_size/2, y + cube_size/2, z1],
            [x - cube_size/2, y + cube_size/2, z1],
        ]
        faces = [
            [0,1,2],[0,2,3], [4,5,6],[4,6,7],
            [0,1,5],[0,5,4], [1,2,6],[1,6,5],
            [2,3,7],[2,7,6], [3,0,4],[3,4,7]
        ]
        meshes.append(trimesh.Trimesh(vertices=vertices, faces=faces))

        if len(meshes) > 1000:
            meshes = [trimesh.util.concatenate(meshes)]

    combined_mesh = trimesh.util.concatenate(meshes) if len(meshes) > 1 else meshes[0]
    combined_mesh.merge_vertices()

    stl_path = os.path.join(current_directory, os.path.splitext(filename)[0] + '.stl')
    combined_mesh.export(stl_path)